# TFPARN (Transformer-based Focal-Pairwise Attentive Ranking Network) for Anti-Spoofing: Complete Technical Documentation

This notebook provides comprehensive documentation of TFPARN (Transformer-based Focal-Pairwise Attentive Ranking Network) and its application in the ASVspoof5 challenge (Audio Anti-Spoofing Detection). The system distinguishes between genuine human speech (bonafide) and AI-generated synthetic speech (spoof).

## 1. Background

### 1.1 ASVspoof5 Challenge Overview

The **ASVspoof5 (Automatic Speaker Verification Spoofing and Countermeasures Challenge 5)** is a competition focused on detecting AI-generated synthetic speech. With the rapid advancement of speech synthesis technology, distinguishing between genuine human speech and sophisticated deepfakes has become critical for security applications.

**Challenge Goal:** Build models that can accurately classify audio samples as:
- **Bonafide (Label=1):** Genuine human speech
- **Spoof (Label=0):** AI-generated synthetic speech

**Key Challenges:**
- Diverse attack types (TTS, VC, various codec compressions)
- Class imbalance in training data
- Need for robust generalization to unseen attack types
- Real-world audio quality variations

### 1.2 Evaluation Metrics

The challenge uses EER (Equal Error Rate), minDCF (Minimum Detection Cost Function), and CLLR (Calibrated Log-Likelihood Ratio) as primary metrics. Lower values indicate better performance.

## 2. Environment Setup

### 2.1 Dependencies

The project requires PyTorch and audio processing libraries:

```
torch~=2.9.0+cu130
numpy~=2.1.2
scikit-learn~=1.7.2
tqdm~=4.67.1
torchaudio~=2.9.0
soundfile~=0.13.1
scipy~=1.15.3
```

**Installation:**
```bash
pip install -r requirements.txt
```

### 2.2 Hardware Requirements

**GPU Memory Requirements (for training):**
- **Batch size 64:** ~8GB VRAM
- **Batch size 96:** ~10GB VRAM  
- **Batch size 128:** ~12GB VRAM

**Recommended Configuration:**
- GPU: NVIDIA RTX 3090 / 4090 or better
- RAM: 32GB+ system memory
- Storage: ~100GB for datasets

**CPU-only mode** is supported but significantly slower for training.

## 3. Model Architecture

### 3.1 Complete Pipeline Overview

The model follows a complete Transformer-based architecture:

```
Raw Waveform -> Log-Mel Spectrogram -> Transformer Encoder -> Pooling -> Classification
```

**Architecture Stages:**

1. **Frontend:** Log-Mel Spectrogram extraction (in-model computation)
2. **Embedding:** Linear projection + Layer Normalization
3. **Positional Encoding:** Sinusoidal positional embeddings
4. **Backbone:** Multi-layer Transformer Encoder (self-attention)
5. **Pooling:** Mean/Attention/Top-k pooling with masking (Attention pooling is used in TFPARN)
6. **Classification Head:** 2-layer MLP -> Binary logits

### 3.2 Model Configuration

```python
from dataclasses import dataclass

@dataclass
class SpeechClassifierArgs:
    # Mel Spectrogram parameters
    n_mels: int = 128          # Number of mel filterbanks
    n_fft: int = 768           # FFT window size
    hop_length: int = 160      # Hop length for STFT
    sample_rate: int = 16000   # Audio sample rate
    
    # Transformer parameters
    d_model: int = 256         # Model dimension
    nhead: int = 8             # Number of attention heads
    num_layers: int = 6        # Number of Transformer layers
    dim_feedforward: int = 1024 # FFN hidden dimension
    dropout: float = 0.3       # Dropout probability
    activation: str = "relu"   # Activation function
    
    # Pooling method: "mean", "attention", "top-k"
    pooling_method: str = "attention"
    top_k_ratio: float = 0.3   # For top-k pooling
```

### 3.3 Architecture Code Example

```python
from model import create_model, SpeechClassifierArgs

# Create model with default configuration
args = SpeechClassifierArgs()
model = create_model(args)

# Or customize architecture
args = SpeechClassifierArgs(
    n_mels=160,
    n_fft=1024,
    d_model=256,
    num_layers=6,
    pooling_method="mean"
)
model = create_model(args)

# Move to device
model = model.to(device)

# Count parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {num_params:,}")
```

### 3.4 Key Architecture Features

**1. In-Model Mel Spectrogram Computation:**
- No need for preprocessing
- Mel filterbank registered as buffer (not trainable)
- Consistent processing during training and inference

**2. Positional Encoding:**
```python
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Sinusoidal positional encoding
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe)
```

**3. Flexible Pooling Strategies:**

- **Mean Pooling:** Average all frame embeddings (default, fastest)
- **Attention Pooling:** Learned attention weights for aggregation
- **Top-k Pooling:** Select top-k frames by L2 norm

**4. Classification Head:**
```python
self.classifier = nn.Sequential(
    nn.Linear(d_model, d_model // 2),
    nn.ReLU(),
    nn.Dropout(dropout),
    nn.Linear(d_model // 2, 2)  # Binary: [spoof, bonafide]
)
```

### 3.5 Forward Pass

```python
# Input: [B, 1, T] raw waveform (mono)
# Output: [B, 2] logits [spoof_score, bonafide_score]

waveforms = batch['waveforms'].to(device)  # [B, 1, 64000]
logits = model(waveforms)  # [B, 2]

# Get predictions
probs = torch.softmax(logits, dim=1)
predictions = torch.argmax(logits, dim=1)  # 0=spoof, 1=bonafide
```

## 4. Data Processing Pipeline

### 4.1 Data Loading Overview

The data pipeline handles:
- Protocol file parsing
- Audio loading (FLAC format)
- Fixed-length processing (crop/repeat strategy)
- RawBoost augmentation (training only)
- Test-Time Augmentation (TTA) for inference

### 4.2 Protocol File Format

ASVspoof5 protocol files contain 10 columns:

```
speaker_id  file_name  gender  codec  codec_q  codec_seed  attack_tag  attack_label  KEY  tmp
```

**Label Mapping:**
- `bonafide` → 1 (genuine human speech)
- `spoof` → 0 (AI-generated speech)

### 4.3 Audio Processing Strategy

**Fixed-Length Processing:**
```python
# Target duration: 4.0 seconds at 16 kHz = 64,000 samples
duration_sec = 4.0
sample_rate = 16000
target_length = int(duration_sec * sample_rate)  # 64,000

# If audio is longer: crop (random for train, center for val/test)
# If audio is shorter: repeat-concatenate then crop
```

**Normalization:**
- Convert to mono (if stereo)
- Normalize amplitude to \[-1, 1\]
- Resample to 16kHz (if needed)

### 4.4 RawBoost Data Augmentation

RawBoost applies various augmentations to improve generalization:

**Algorithm 1: Linear/Nonlinear Convolution**
```python
# Generate random FIR filter
N_fir = np.random.randint(5, 15)
h = np.random.randn(N_fir)
h = h / np.sum(np.abs(h))

# Apply convolution
x_conv = signal.convolve(x, h, mode='same')

# Optional nonlinear distortion
if np.random.rand() > 0.5:
    alpha = np.random.uniform(0.1, 0.5)
    x_conv = np.tanh(alpha * x_conv)
```

**Algorithm 2: IIR Filtering**
```python
# Randomly select filter type
filter_type = np.random.choice(['lowpass', 'highpass', 'bandpass'])

# Apply butterworth filter
if filter_type == 'lowpass':
    cutoff = np.random.uniform(1000, 4000)  # Hz
    b, a = signal.butter(4, cutoff / (sample_rate / 2), btype='low')
```

**Algorithm 3: Additive Noise**
```python
# Add stationary noise with random SNR (10-40 dB)
snr_db = np.random.uniform(10, 40)
noise = np.random.randn(len(x))
x_noisy = x + noise * np.sqrt(signal_power / (10 ** (snr_db / 10)))
```

### 4.5 Test-Time Augmentation (TTA)

TTA generates multiple crops per sample for variance reduction:

```python
# For inference: generate 5 overlapping crops per audio
# Model predicts on all crops, then averages logits

class TTADataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, num_crops=5):
        self.base_dataset = base_dataset
        self.num_crops = num_crops
    
    def __getitem__(self, index):
        # Generate multiple crops with 50% overlap
        waveforms = self.base_dataset.generate_tta_crops(
            waveform, num_crops=self.num_crops
        )  # [num_crops, C, T]
        return {
                "waveforms": waveforms,  # [num_crops, C, T]
                "length": waveforms.shape[-1],
                "label": label,
                "speaker_id": item['speaker_id'],
                "attack_label": item['attack_label'],
                "audio_path": str(audio_path)
        }

# During inference
logits_crops = model(waveforms_flat)  # [B*num_crops, 2]
logits = logits_crops.view(B, num_crops, 2).mean(dim=1)  # Average
```

### 4.6 DataLoader Creation

```python
from data_process import make_loaders, DefaultArgs

# Configure data loading
args = DefaultArgs()
args.train_data_dir = "path/to/train/flac/"
args.train_protocol_dir = "path/to/train.tsv"
args.batch_size = 96
args.num_workers = 8
args.use_rawboost = True  # Enable RawBoost for training
args.rawboost_prob = 0.5  # Apply to 50% of samples
args.use_tta = True       # Enable TTA for dev/eval
args.tta_num_crops = 5    # Number of crops per sample

# Create dataloaders
train_loader, dev_loader, eval_loader = make_loaders(args)

# Batch structure
for batch in train_loader:
    waveforms = batch['waveforms']  # [B, 1, 64000]
    labels = batch['labels']        # [B]
    lengths = batch['lengths']      # [B]
    break
```

## 5. Training Pipeline

### 5.1 Training Configuration

```python
from dataclasses import dataclass

@dataclass
class ModelArgs:
    # ...

    # Training hyperparameters
    max_epochs: int = 80
    batch_size: int = 96
    learning_rate: float = 1e-4
    weight_decay: float = 1e-2
    optimizer_type: str = "adamw"  # 'adam' or 'adamw'
    
    # Scheduler
    scheduler_type: str = "cosine"  # 'cosine', 'step', or 'none'
    scheduler_warmup_epochs: int = 5
    
    # Loss function
    loss_type: str = "focal"  # 'ce' or 'focal'
    focal_alpha: float = 0.1   # Weight for positive class
    focal_gamma: float = 2.0   # Focusing parameter
    
    # Pairwise ranking loss
    enable_pairwise: bool = True
    pairwise_margin: float = 1.0
    pairwise_weight: float = 0.3
    
    # Early stopping
    early_stopping_patience: int = 15
    early_stopping_metric: str = "eer"  # 'eer', 'f1_macro', 'accuracy'
    early_stopping_mode: str = "min"    # 'min' for eer, 'max' for f1/acc

    # ...
```

### 5.2 Focal Loss for Class Imbalance

Focal Loss addresses class imbalance by down-weighting easy examples:

```python
class FocalLoss(nn.Module):
    """
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    Args:
        alpha: Weighting factor [num_classes]
        gamma: Focusing parameter (default: 2.0)
    """
    def __init__(self, alpha: torch.Tensor, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, logits, labels):
        probs = F.softmax(logits, dim=1)
        probs_t = (probs * F.one_hot(labels, num_classes=2).float()).sum(dim=1)
        
        alpha_t = (self.alpha * F.one_hot(labels, num_classes=2).float()).sum(dim=1)
        focal_weight = alpha_t * (1 - probs_t) ** self.gamma
        ce_loss = F.cross_entropy(logits, labels, reduction='none')
        
        return (focal_weight * ce_loss).mean()

# Usage
focal_alpha = torch.tensor([0.9, 0.1])  # [spoof_weight, bonafide_weight]
criterion = FocalLoss(focal_alpha, gamma=2.0)
```

**Why Focal Loss?**
- ASVspoof5 training data is imbalanced (more spoof samples)
- Focal Loss focuses on hard-to-classify examples

### 5.3 Pairwise Ranking Loss

Optimizes ranking-based metrics (EER, minDCF):

```python
class PairwiseRankingLoss(nn.Module):
    """
    Encourages bonafide samples to score higher than spoof samples
    Loss = max(0, margin - (score_bonafide - score_spoof))
    """
    def __init__(self, margin: float = 1.0):
        super().__init__()
        self.margin = margin
    
    def forward(self, logits, labels):
        scores = logits[:, 1]  # Bonafide scores
        
        bonafide_scores = scores[labels == 1]
        spoof_scores = scores[labels == 0]
        
        # Create pairwise differences
        score_diff = bonafide_scores[:, None] - spoof_scores[None, :]
        pairwise_loss = F.relu(self.margin - score_diff)
        
        return pairwise_loss.mean()

# Combined loss
total_loss = focal_loss + 0.3 * pairwise_loss
```

### 5.4 Optimizer and Scheduler

```python
# AdamW optimizer with weight decay
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

# Cosine annealing scheduler with linear warmup
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max_epochs - warmup_epochs,
    eta_min=1e-6
)

# Linear warmup for first 5 epochs
for epoch in range(1, warmup_epochs + 1):
    warmup_lr = learning_rate * epoch / warmup_epochs
    for param_group in optimizer.param_groups:
        param_group['lr'] = warmup_lr
```

### 5.5 Training Loop

```python
from utils import EarlyStopping, compute_all_metrics

# Early stopping
early_stopping = EarlyStopping(patience=15, mode='min')

best_eer = float('inf')

for epoch in range(1, max_epochs + 1):
    # Training
    model.train()
    for batch in train_loader:
        waveforms = batch['waveforms'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        logits = model(waveforms)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
    
    # Validation with TTA
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in dev_loader:
            waveforms = batch['waveforms'].to(device)  # [B, num_crops, C, T]
            labels = batch['labels'].to(device)
            
            # TTA: reshape and average
            B, num_crops, C, T = waveforms.shape
            waveforms_flat = waveforms.view(B * num_crops, C, T)
            logits_flat = model(waveforms_flat)
            logits = logits_flat.view(B, num_crops, 2).mean(dim=1)
            
            all_logits.append(logits.cpu())
            all_labels.append(labels.cpu())
    
    # Compute metrics
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    metrics = compute_all_metrics(all_logits, all_labels)
    
    print(f"Epoch {epoch}: EER={metrics['eer']:.4f}, minDCF={metrics['min_dcf']:.4f}")
    
    # Save best model
    if metrics['eer'] < best_eer:
        best_eer = metrics['eer']
        torch.save(model.state_dict(), 'best_model.pt')
    
    # Early stopping
    if early_stopping(metrics['eer']):
        print(f"Early stopping at epoch {epoch}")
        break
    
    # Update scheduler
    if epoch > warmup_epochs:
        scheduler.step()
```

## 6. Evaluation Pipeline

### 6.1 Overview

The evaluation pipeline computes standard metrics (EER, minDCF, actDCF) and applies Platt calibration for score normalization:

```python
from utils import compute_all_metrics, apply_platt_calibration

# Get predictions
dev_logits, dev_labels = evaluate_model(model, dev_loader, device)
eval_logits, eval_labels = evaluate_model(model, eval_loader, device)

# Apply Platt calibration (fit on dev, apply to eval)
calibrated_scores, _ = apply_platt_calibration(
    dev_logits, dev_labels, eval_logits
)

# Compute metrics
metrics = compute_metrics_from_scores(calibrated_scores, eval_labels)
print(f"EER: {metrics['eer']:.4f}, minDCF: {metrics['min_dcf']:.4f}")
```

### 6.2 ASVspoof5 Track 1 Metrics

**minDCF (Minimum Detection Cost Function):**
```python
def compute_min_dcf(scores, labels, c_miss=1.0, c_fa=10.0, p_target=0.05):
    """
    Normalized minimum DCF for ASVspoof5 Track 1
    Parameters: C_miss=1.0, C_fa=10.0, π_spf=0.05
    """
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    
    dcf = c_miss * fnr * p_target + c_fa * fpr * (1 - p_target)
    dcf_def = min(c_miss * p_target, c_fa * (1 - p_target))
    
    return np.min(dcf / dcf_def)
```

**actDCF (Actual Detection Cost Function):**
Computed at Bayes-optimal threshold: τ_bayes = -log(β), where β ≈ 1.90 for ASVspoof5 Track
```python
def compute_act_dcf(
    scores: np.ndarray,
    labels: np.ndarray,
    c_miss: float = 1.0,
    c_fa: float = 10.0,
    p_target: float = 0.05
) -> float:
    """
    Compute actual Detection Cost Function (actDCF) at Bayes threshold for ASVspoof5 Track 1

    Following ASVspoof5 specification:
    - τ_bayes = -log(β) where β = C_miss * (1 - π_spf) / (C_fa * π_spf) ≈ 1.90
    - actDCF = DCF'(τ_bayes) (normalized actual DCF at Bayes-optimal threshold)

    Note: This assumes detection scores can be interpreted as log-likelihood ratios.
    If scores are probabilities, conversion may be needed.

    Args:
        scores: Prediction scores (higher = more likely bonafide)
        labels: Ground truth labels (0=spoof, 1=bonafide)
        c_miss: Cost of missing a bonafide (false negative), default=1.0
        c_fa: Cost of false alarm on spoof (false positive), default=10.0
        p_target: Prior probability of bonafide (1 - π_spf), default=0.05

    Returns:
        act_dcf_normalized: Normalized actual DCF at Bayes threshold
    """
    # Compute β (beta factor)
    beta = (c_miss * p_target) / (c_fa * (1 - p_target))

    # Bayes-optimal threshold τ_bayes = -log(β)
    # For probability scores in [0,1], we need to convert to log-likelihood ratios
    # Since scores are probabilities P(bonafide|x), we compute log-odds
    eps = 1e-10
    scores_clipped = np.clip(scores, eps, 1 - eps)

    # Convert probability scores to log-likelihood ratios
    # LLR = log(P(bonafide|x) / P(spoof|x))
    llr_scores = np.log(scores_clipped / (1 - scores_clipped))

    # Bayes threshold in log-likelihood ratio space
    tau_bayes = -np.log(beta)

    # Make predictions at Bayes threshold
    predictions = (llr_scores >= tau_bayes).astype(int)

    # Compute confusion matrix elements
    tp = np.sum((predictions == 1) & (labels == 1))
    fp = np.sum((predictions == 1) & (labels == 0))
    fn = np.sum((predictions == 0) & (labels == 1))
    tn = np.sum((predictions == 0) & (labels == 0))

    # Compute error rates
    fnr = fn / (tp + fn + 1e-10)  # P_miss (miss rate for bonafide)
    fpr = fp / (fp + tn + 1e-10)  # P_fa (false alarm rate for spoof)

    # Compute unnormalized actual DCF
    act_dcf = c_miss * fnr * p_target + c_fa * fpr * (1 - p_target)

    # Normalize by DCF_def
    dcf_def = min(c_miss * p_target, c_fa * (1 - p_target))
    act_dcf_normalized = act_dcf / dcf_def

    return act_dcf_normalized
```

### 6.3 Complete Evaluation

```python
def evaluate_with_calibration(model, dev_loader, eval_loader, device):
    """Evaluation with calibration and prior correction"""
    # Get predictions
    dev_logits, dev_labels = evaluate_model(model, dev_loader, device, use_tta=True)
    eval_logits, eval_labels = evaluate_model(model, eval_loader, device, use_tta=True)
    
    # Apply calibration
    calibrated_scores, _ = apply_platt_calibration(dev_logits, dev_labels, eval_logits)
    corrected_scores = apply_prior_correction(dev_labels, eval_labels, calibrated_scores)
    
    # Compute metrics
    metrics = compute_metrics_from_scores(corrected_scores, eval_labels)
    metrics['cllr'] = compute_cllr(corrected_scores, eval_labels)
    
    return metrics
```

For detailed implementation of calibration methods and CLLR computation, see `utils.py`.

## 7. Project Structure

```
true_tone3/
│
├── model.py                    # Transformer model architecture
│   ├── SpeechClassifierArgs    # Model configuration dataclass
│   ├── PositionalEncoding      # Sinusoidal positional encoding
│   ├── SpeechTransformerClassifier  # Main model class
│   └── create_model()          # Model factory function
│
├── data_process.py             # Data loading and preprocessing
│   ├── DefaultArgs             # Data loading configuration
│   ├── RawBoost                # RawBoost augmentation class
│   ├── read_protocol()         # Parse ASVspoof5 protocol files
│   ├── ASV5Dataset             # PyTorch Dataset class
│   ├── TTADataset              # Test-Time Augmentation wrapper
│   ├── collate_fn()            # Batch collation function
│   └── make_loaders()          # DataLoader creation
│
├── utils.py                          # Utility functions
│   ├── set_seed()                    # Random seed fixing
│   ├── get_device()                  # Device management
│   ├── FocalLoss                     # Focal loss for class imbalance
│   ├── PairwiseRankingLoss           # Pairwise ranking loss
│   ├── CombinedLoss                  # Combined loss wrapper
│   ├── compute_eer()                 # Equal Error Rate
│   ├── compute_min_dcf()             # Minimum Detection Cost Function
│   ├── compute_cllr()                # Calibrated Log-Likelihood Ratio
│   ├── apply_platt_calibration()     # Platt calibration
│   ├── apply_prior_correction()      # Prior correction
│   ├── evaluate_model()              # Model evaluation
│   ├── evaluate_with_calibration()   # Complete evaluation pipeline
│   ├── load_model_weights()          # Model checkpoint loading
│   ├── save_model()                  # Model checkpoint saving
│   └── EarlyStopping                 # Early stopping handler
│
├── main_train.py                     # Main training script
│   ├── ModelArgs                     # Complete training configuration
│   ├── train_one_epoch()             # Training loop for one epoch
│   ├── validate()                    # Validation with TTA
│   └── main()                        # Main training pipeline
│
├── read_and_evaluate.py              # Model evaluation script
│   ├── DatasetConfig                 # Dataset configuration
│   ├── EvaluationConfig              # Evaluation configuration
│   ├── create_dataloader()           # Create single dataloader
│   ├── evaluate_dataset()            # Evaluate on single dataset
│   └── main()                        # Main evaluation pipeline
│
├── run_multiple_experiments.py # Hyperparameter tuning script
│   └── Grid search over hyperparameters
│
├── requirements.txt                  # Python dependencies
│
├── README.md                         # Quick start guide
│
└── Introduction_of_true_tone5.ipynb  # This file
```

### 7.1 Core Modules

**model.py:**
- Complete Transformer architecture
- In-model mel spectrogram computation
- Flexible pooling strategies
- ~4.8M trainable parameters

**data_process.py:**
- ASVspoof5 protocol parsing
- Audio loading and preprocessing
- RawBoost augmentation
- Test-Time Augmentation support

**utils.py:**
- Comprehensive evaluation metrics
- Focal Loss + Pairwise Ranking Loss
- Platt calibration and prior correction

### 7.2 Scripts

**main_train.py:**
- End-to-end training pipeline
- Automatic device selection
- Early stopping
- Best model saving
- Final evaluation with calibration

**read_and_evaluate.py:**
- Flexible evaluation on multiple datasets
- Automatic calibration (uses Dev as reference)
- Supports TTA
- Detailed metrics reporting

**run_multiple_experiments.py:**
- Grid search over hyperparameters
- Automated experiment tracking
- Parallel experiment support

## 8. Technical Summary

### 8.1 Key Innovations

**1. Focal Loss for Class Imbalance**
- Addresses class imbalance in ASVspoof5 training data
- Down-weights easy examples, focuses on hard-to-classify samples
- Focal Loss: `FL(p_t) = -α_t * (1 - p_t)^γ * log(p_t)`
- Significant improvement over standard Cross-Entropy
- Hyperparameters: α=0.1 (bonafide weight), γ=2.0 (focusing parameter)

**2. Pairwise Ranking Loss**
- Directly optimizes ranking-based metrics (EER/minDCF)
- Encourages bonafide samples to score higher than spoof samples
- Loss: `L_pairwise = max(0, margin - (score_bonafide - score_spoof))`
- Combined with Focal Loss: `L_total = L_focal + λ * L_pairwise`
- Hyperparameters: margin=1.0, λ=0.3 (pairwise weight)

**3. Attention Pooling**
- Learned attention mechanism for frame-level aggregation
- Automatically weights informative frames higher than uninformative frames
- More effective than mean pooling or top-k pooling
- Attention scores: `α_t = softmax(w^T * tanh(W * h_t))`
- Aggregated representation: `h = Σ α_t * h_t`

**4. Test-Time Augmentation**
- Generate 5 overlapping crops per sample during inference
- Average logits across crops for robust predictions
- Reduces prediction variance caused by random cropping
- ~5% improvement in EER

**5. RawBoost Augmentation**
- Three augmentation algorithms (convolution, filtering, noise)
- Applied during training only with 50% probability
- Improves generalization to unseen attacks and codec variations
- Prevents overfitting to training data distribution

### 8.2 Architecture Highlights

**Transformer Configuration:**
```
Input: [B, 1, 64000] → 16kHz, 4-second waveform
Frontend: Log-Mel Spectrogram [B, T', 160]
Embedding: Linear projection [B, T', 256]
Positional: Sinusoidal encoding
Backbone: 6-layer Transformer (8 heads, dim=256)
Pooling: Mean/Attention/Top-k (Attention is used in TFPARN)
Classifier: 2-layer MLP → [B, 2] logits
```

**Model Size:**
- Parameters: ~4.8M trainable
- VRAM: 8-12GB (batch size dependent)

**Training Time:**
- 1 epoch(Training + Validating with TTA): ~5 minutes (RTX 5090)
- Full training: 4-8 hours (with early stopping)
- Batch size 96: optimal for 10GB VRAM

### 8.3 Advantages and Limitations

**Advantages:**

✓ End-to-end trainable (no preprocessing)

✓ Strong generalization with Focal Loss

✓ Calibrated probability outputs

✓ Flexible pooling strategies

✓ TTA for improved robustness

✓ Complete evaluation pipeline


**Limitations:**

✗ Requires GPU for efficient training (CPU can be used, but very slow)

✗ Memory-intensive for large batch sizes

✗ No explicit codec awareness

## 10. References

### 10.1 Datasets

- ASVspoof 2021 Dataset: https://www.kaggle.com/datasets/mohammedabdeldayem/avsspoof-2021
- ASVspoof 2019 Database: https://www.kaggle.com/datasets/awsaf49/asvpoof-2019-dataset
- ASVspoof 5: https://zenodo.org/records/14498691

### 10.2 Key Papers

1. **RawBoost:**
   - "RawBoost: A Raw Data Boosting and Augmentation Method applied to Automatic Speaker Verification Anti-Spoofing" (ICASSP 2022)
   - Three augmentation algorithms (implemented in this project)
   - Paper: https://arxiv.org/abs/2111.04433

2. **Focal Loss:**
   - "Focal Loss for Dense Object Detection" (ICCV 2017)
   - Addresses class imbalance by down-weighting easy examples
   - Paper: https://arxiv.org/abs/1708.02002

3. **Attention Is All You Need:**
   - "Attention Is All You Need" (NeurIPS 2017)
   - Original Transformer architecture
   - Paper: https://arxiv.org/abs/1706.03762

4. **Platt Calibration:**
   - "Probabilistic Outputs for Support Vector Machines and Comparisons to Regularized Likelihood Methods" (1999)
   - Sigmoid-based probability calibration method
   - Paper: https://www.cs.colorado.edu/~mozer/Teaching/syllabi/6622/papers/Platt1999.pdf

### 10.3 Metrics and Evaluation

**EER (Equal Error Rate):**
- Standard metric for biometric systems
- Point where FPR = FNR
- Lower is better

**minDCF (Minimum Detection Cost Function):**
- Weighted combination of miss and false alarm rates
- Standard in speaker verification
- Formula: `DCF = C_miss * P_miss * P_target + C_fa * P_fa * (1 - P_target)`

**CLLR (Calibrated Log-Likelihood Ratio):**
- Measures probability calibration quality
- CLLR = 0: Perfect calibration
- CLLR > 1: Poor calibration
- Reference: "Application-Independent Evaluation of Speaker Detection" (Computer Speech & Language 2006)
- Paper: https://www.sciencedirect.com/science/article/abs/pii/S0885230805000306
